In [ ]:
# Day 3 - Section 3.6.2  --  "CAI, With the Magic Removed"
# ---------------------------------------------------------------------------
# WHAT THIS NOTEBOOK IS: a complete tool-calling AI agent in ~40 lines.
# The ENTIRE agent is Cells 2-4. Read them before you run anything (Beat 1).
# CAI, ShellGPT and PentestGPT are this same idea with more tools + better prompts.
# ---------------------------------------------------------------------------

# Install the one library we need, pinned so the lab behaves identically for everyone.
#   openai : the official OpenAI Python client. We use ONE call - chat.completions.create -
#            which is what lets the model ask to run our tools ("function calling").
!pip -q install openai==1.51.0

import json                         # stdlib: turn tool results into text for the model, and back
from openai import OpenAI           # the client class we talk to the model through
from google.colab import userdata   # Colab's secret store - keeps the API key OUT of the notebook

# The key is READ from Colab Secrets (left sidebar > key icon > OPENAI_API_KEY).
# It is never typed here, never printed, never saved in the file. That is the rule every lab uses.
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))   # shared, spend-capped course key

MODEL     = "gpt-4o-mini"   # pinned at build time (Day 3 deck v15). Cheapest reliable tool-caller.
MAX_STEPS = 12              # the brake. A loop with no brake is how agents run up a bill.


In [ ]:
# CELL 2 - THE TOOLS. THEY ARE DICTIONARIES.
# ---------------------------------------------------------------------------
# This is the SAME Acme Financial Services estate you triaged in Day 2 Lab 6.
# You already have the answer key. Your job today is to GRADE the agent, not admire it.
# Nothing here reaches the network: the "estate" and the "CVE feed" are just Python dicts,
# so the lab cannot fail because a website is down.
# ---------------------------------------------------------------------------

ESTATE = {
  "web-01":    {"os": "Debian 12 stable", "xz": "5.4.1", "internet_facing": True,  "seg": "DMZ"},
  "build-02":  {"os": "Debian sid",       "xz": "5.6.1", "internet_facing": True,  "seg": "DMZ"},
  "app-03":    {"os": "Ubuntu 22.04",     "xz": "5.2.5", "internet_facing": True,  "seg": "AWS VPC"},
  "ci-04":     {"os": "Fedora 40",        "xz": "5.6.0", "internet_facing": False, "seg": "INTERNAL"},
  "db-05":     {"os": "RHEL 9",           "xz": "5.2.5", "internet_facing": False, "seg": "INTERNAL"},
  "ws-fin-11": {"os": "Windows 11",       "xz": None,    "internet_facing": False, "seg": "INTERNAL"},
  "ws-fin-12": {"os": "Windows 11",       "xz": None,    "internet_facing": False, "seg": "INTERNAL"},
  "ws-eng-07": {"os": "Windows 11",       "xz": None,    "internet_facing": False, "seg": "INTERNAL"},
}

CVES = {
  "CVE-2024-3094": {"package": "xz-utils", "affected": ["5.6.0", "5.6.1"], "cvss": 10.0,
                    "note": "Backdoor in the upstream tarball. Only these two versions shipped it."},
}

# Three functions. Each one is a dictionary lookup - nothing in computing is more deterministic.
def list_hosts():           return list(ESTATE)                                   # every host name
def lookup_host(hostname):  return ESTATE.get(hostname, {"error": "no such host"}) # facts for one host
def lookup_cve(cve_id):     return CVES.get(cve_id,     {"error": "unknown CVE"})  # facts for one CVE

# DISPATCH maps the name the model asks for -> the real Python function that answers it.
DISPATCH = {"list_hosts": list_hosts, "lookup_host": lookup_host, "lookup_cve": lookup_cve}


In [ ]:
# CELL 3 - THE ALLOW-LIST.
# ---------------------------------------------------------------------------
# THIS ARRAY IS THE ALLOW-LIST. The agent can do these three things and NOTHING else.
# Each entry describes one tool to the model: its name, what it does, and what arguments
# it takes. The model reads these descriptions to decide which tool to call and with what.
# Remember this cell on Friday afternoon (Day 5, "Governing an Agent, Not a Model").
# ---------------------------------------------------------------------------

TOOLS = [
  {"type": "function", "function": {
      "name": "list_hosts",
      "description": "Return the names of every host in the estate.",
      "parameters": {"type": "object", "properties": {}, "required": []}}},

  {"type": "function", "function": {
      "name": "lookup_host",
      "description": "Return OS, xz-utils version, internet exposure and network segment for one host.",
      "parameters": {"type": "object", "properties": {
          "hostname": {"type": "string", "description": "Exact host name, e.g. build-02"}},
          "required": ["hostname"]}}},

  {"type": "function", "function": {
      "name": "lookup_cve",
      "description": "Return package, affected version list and CVSS score for a CVE id.",
      "parameters": {"type": "object", "properties": {
          "cve_id": {"type": "string", "description": "e.g. CVE-2024-3094"}},
          "required": ["cve_id"]}}},
]


In [ ]:
# CELL 4 - THE AGENT. THIS IS THE WHOLE THING.
# ---------------------------------------------------------------------------
# A dictionary of tools (Cell 2), a list describing them (Cell 3), and the loop below.
# Each time round the loop: ask the model -> if it asked for a tool, run the tool and hand
# back the result -> repeat, until the model decides it is finished. That is an agent.
# ---------------------------------------------------------------------------

def run_agent(question, tools=TOOLS):
    messages = [{"role": "user", "content": question}]   # the running transcript

    for step in range(MAX_STEPS):                        # MAX_STEPS is the brake from Cell 1
        r = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        m = r.choices[0].message
        messages.append(m)                               # remember what the model just said

        # The model stops asking for tools when it is ready to answer. Nobody scripted when.
        if r.choices[0].finish_reason != "tool_calls":
            print("\n" + "="*60 + "\nFINAL ANSWER\n" + "="*60)
            print(m.content)
            return m.content

        # Otherwise it asked for one or more tools. It CHOSE these. Nobody scripted them.
        for call in m.tool_calls:
            args   = json.loads(call.function.arguments)          # the model's chosen arguments
            result = DISPATCH[call.function.name](**args)         # run the real, deterministic tool
            print(f"  TOOL  {call.function.name}({args})  ->  {result}")
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": json.dumps(result)})      # hand the result back to the model

    print("hit MAX_STEPS - the brake worked")                     # safety net if it never stops

# Run it. Watch the TOOL lines scroll - the order is the model's, not ours.
run_agent("CVE-2024-3094 just dropped. Which hosts must we patch TONIGHT, and why?")


In [ ]:
# CELL 5 - BEAT 4. BREAK THE ALLOW-LIST.
# ---------------------------------------------------------------------------
# We add a FOURTH tool that does something in the world (opens a change ticket), then watch:
#   Run A: the tool is available but the agent does not use it  -> a capability is not a behaviour
#   Run B: one extra clause in the QUESTION makes it fire       -> nothing was hacked; it used a tool it was given
#   Run C: delete the tool from the list and the ability is gone -> that is the allow-list
# ---------------------------------------------------------------------------

FIRED = []                                    # a record of every ticket actually raised
def create_change_ticket(host, action):
    FIRED.append((host, action))
    print(f"  *** TICKET RAISED: {host} -> {action}")
    return {"ticket": f"CHG-{1000+len(FIRED)}", "status": "OPEN"}

DISPATCH["create_change_ticket"] = create_change_ticket   # wire the new tool into dispatch

TICKET_TOOL = {"type": "function", "function": {
    "name": "create_change_ticket",
    "description": "Open a change ticket to patch a host. This performs a REAL action in the change system.",
    "parameters": {"type": "object", "properties": {
        "host":   {"type": "string"},
        "action": {"type": "string"}},
        "required": ["host", "action"]}}}

# RUN A - same question, four tools available. Does it fire the ticket tool?  (Expect: no.)
run_agent("CVE-2024-3094 just dropped. Which hosts must we patch TONIGHT, and why?",
          tools = TOOLS + [TICKET_TOOL])

# RUN B - one clause added to the question. Watch what changes.  (Expect: it fires the ticket.)
run_agent("CVE-2024-3094 just dropped. Which hosts must we patch TONIGHT, and why? "
          "Make sure it actually gets actioned.",
          tools = TOOLS + [TICKET_TOOL])

# RUN C - delete the tool from the allow-list. Same question as Run B.  (Expect: it cannot.)
run_agent("CVE-2024-3094 just dropped. Which hosts must we patch TONIGHT, and why? "
          "Make sure it actually gets actioned.",
          tools = TOOLS)              # <- TICKET_TOOL is simply not in the list
